# Stock Price Prediction

## Problem Statement
Stock price prediction is a complex task due to the high volatility and randomness of financial markets. Traditional statistical models often fail to capture long-term dependencies in stock price movements. Deep learning, particularly Long Short-Term Memory (LSTM) networks, has shown great potential in handling time-series forecasting by learning from historical patterns.

## Step-by-Step

## Precheck

In [3]:
%%time
%pip install ipykernel ipython-autotime ipywidgets -U --force-reinstall --break-system-packages --no-warn-script-location 

%load_ext autotime

  Using cached ipykernel-6.29.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached ipython_autotime-0.3.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached ipywidgets-8.1.6-py3-none-any.whl.metadata (2.4 kB)
  Using cached appnope-0.1.4-py2.py3-none-any.whl.metadata (908 bytes)
  Using cached comm-0.2.2-py3-none-any.whl.metadata (3.7 kB)
  Using cached debugpy-1.8.14-cp313-cp313-macosx_14_0_universal2.whl.metadata (1.3 kB)
  Using cached ipython-9.1.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached jupyter_client-8.6.3-py3-none-any.whl.metadata (8.3 kB)
  Using cached jupyter_core-5.7.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached matplotlib_inline-0.1.7-py3-none-any.whl.metadata (3.9 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached psutil-7.0.0-cp36-abi3-macosx_11_0_arm64.whl.metadata (22 kB)
  Using cached pyzmq-26.4.0-cp313-cp313-macosx_10_15_universal2.whl.metadata (6.0

In [11]:
%%time
!python3 -V && pip3 -V

Python 3.13.3
pip 25.0.1 from /opt/homebrew/lib/python3.13/site-packages/pip (python 3.13)
CPU times: user 3.48 ms, sys: 10.8 ms, total: 14.3 ms
Wall time: 481 ms
time: 482 ms (started: 2025-04-16 08:24:44 -07:00)


### Install dependencies

In [ ]:
!pip3 install -U yfinance torch numpy pandas scikit-learn matplotlib ipython-autotime pip-system-certs certifi 

%load_ext autotime

### Import Libraries 

In [5]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yfinance as yf   # ref: https://yfinance-python.org/
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

### Load & Preprocess Stock Data

In [6]:
# BEGIN: fix Python or Notebook SSL CERTIFICATE_VERIFY_FAILED
import os, ssl
# BEGIN: fix Python or Notebook SSL CERTIFICATE_VERIFY_FAILED
if (not os.environ.get('PYTHONHTTPSVERIFY', '') and getattr(ssl, '_create_unverified_context', None)):
    ssl._create_default_https_context = ssl._create_unverified_context
# END: fix Python or Notebook SSL CERTIFICATE_VERIFY_FAILED

#### Define variables

In [7]:
# Download stock data for Apple (AAPL)
stock='AAPL'
todaysDate = pd.Timestamp.today().date().strftime('%Y-%m-%d')

print(f"Downloading stock data for '{stock}' from '1990-01-01' to '{todaysDate}' ")

# yf.enable_debug_mode()
# data = yf.download(stock) # start='1990-01-01', end=todaysDate)
data = yf.Ticker(stock).history(period='max')
data

In [44]:

# Extract 'Close' prices and scale them
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data[['Close']])

# Create time-series sequences
def create_sequences(data, seq_length=50):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 50  # Lookback period
X, y = create_sequences(data_scaled, seq_length)

# Train-test split (80-20)
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Convert to PyTorch tensors
X_train_tensor, X_test_tensor = torch.Tensor(X_train), torch.Tensor(X_test)
y_train_tensor, y_test_tensor = torch.Tensor(y_train), torch.Tensor(y_test)
